# One constraint spec, three models

A visual comparison of the three fitted-joint engines on a *hand-specified*
distribution over four variables `x0..x3` on `[0, 100]` — the same spec
translated into each model's native constraint grammar:

| # | constraint | type |
|---|---|---|
| 1 | `E[x0] = 65` | expectation |
| 2 | `E[x2] = 49` | expectation |
| 3 | `E[x3] = 50` | expectation |
| 4 | `P(x1 > 60) = 0.30` | tail indicator |
| 5 | `P(x1 > 60 \| x0 > 70) = 0.60` | conditional probability |
| 6 | `P(x1 < 30 \| x0 < 40) = 0.55` | conditional probability |
| 7 | `E[x2 \| x1 > 60] = 70` | conditional expectation |
| 8 | `E[x2 \| x1 < 60] = 40` | conditional expectation |
| 9 | `cov(x2, x3) = -250` | covariance |
| 10 | `P(x0 > 70, x3 > 70) = 0.15` | joint tail indicator |

The set is (mostly) self-consistent — #2 is the total-probability anchor for
#4/#7/#8 (`0.30*70 + 0.70*40 = 49`) — and its dependency graph is a **loop**:

```
x0 —(5,6)— x1 —(7,8)— x2 —(9)— x3 —(10)— x0
```

The tensor chain is ordered `x0-x1-x2-x3`, so the closing `x0–x3` edge must be
carried through every bond — the structural stress. The `x1->x2` push is
positive while `cov(x2,x3) < 0` yet `x0,x3` are positively tied, so the loop is
also mildly *frustrated*: no joint satisfies everything exactly, and where each
model spends its residual is part of what the plots show.

**Models** (one color each, constant across every figure):
<span style="color:#7f7f7f">**gray** = independent</span> (bond-1 tensor chain —
the best *product* distribution for this spec),
<span style="color:#ff7f0e">**orange** = tensor chain</span> (Born, bond 8,
exact contractions), <span style="color:#1f77b4">**blue** = flow</span>
(RealNVP sampler, exact-entropy maxent).

In [ ]:
import os
os.environ.setdefault("JAX_PLATFORMS", "cpu")
import time
import numpy as np
import matplotlib.pyplot as plt

from calibrated_response.tn import ContinuousVar
from calibrated_response.tn.chain import TensorChain
from calibrated_response.tn.losses import combined_loss
from calibrated_response.maxent_sampler import (
    FlowSamplerModel, moment, soft_gt, soft_lt)

NBINS, SPAN, NV = 20, 100.0, 4
vars_ = [ContinuousVar(f"x{i}", 0.0, 100.0, NBINS) for i in range(NV)]
names = [v.name for v in vars_]
centers = (np.arange(NBINS, dtype=np.float32) + 0.5) * (SPAN / NBINS)

COLOR = {"independent": "#7f7f7f", "tensor chain": "#ff7f0e", "flow": "#1f77b4"}
CMAP = {"independent": "Greys", "tensor chain": "Oranges", "flow": "Blues"}

# ---- the spec, in the tensor-network grammar (hard bin masks; thresholds
# ---- sit on bin edges, so the masks are exact) -------------------------
mgt = lambda thr: (centers > thr).astype(np.float32)
mlt = lambda thr: (centers < thr).astype(np.float32)
W_E = (1.0 / SPAN) ** 2                # normalise expectation errors to spans
W_XY = (1.0 / 2500.0) ** 2             # ... and E[x*y] errors to (span/2)^2

tn_csts = [
    ("expect", {0: centers}, 65.0, W_E),
    ("expect", {2: centers}, 49.0, W_E),
    ("expect", {3: centers}, 50.0, W_E),
    ("prob", {1: mgt(60)}, 0.30),
    ("cond", {1: mgt(60)}, {0: mgt(70)}, 0.60),
    ("cond", {1: mlt(30)}, {0: mlt(40)}, 0.55),
    ("cond_expect", 2, {1: mgt(60)}, 70.0, W_E),
    ("cond_expect", 2, {1: mlt(60)}, 40.0, W_E),
    # cov via the uncentred moment: E[x2*x3] = cov + E[x2]E[x3] = -250 + 49*50
    ("expect", ((2, 3), np.outer(centers, centers)), 2200.0, W_XY),
    ("prob", {0: mgt(70), 3: mgt(70)}, 0.15),
]

# ---- the same spec in the flow-sampler grammar (soft indicators inside the
# ---- loss only; sharpness 0.3 on the [0,100] domain; w = 1/(2 sd^2)) ----
SH = 0.3
W_P, W_EF, W_C = 200.0, 0.02, 3.2e-5   # sd: 0.05 (prob), 5 (E), 125 (cov)
conj = lambda f, g: (lambda x: f(x) * g(x))

fl_csts = [
    ("expect", moment(0), 65.0, W_EF),
    ("expect", moment(2), 49.0, W_EF),
    ("expect", moment(3), 50.0, W_EF),
    ("expect", soft_gt(1, 60.0, SH), 0.30, W_P),
    ("cond_expect", soft_gt(1, 60.0, SH), soft_gt(0, 70.0, SH), 0.60, W_P),
    ("cond_expect", soft_lt(1, 30.0, SH), soft_lt(0, 40.0, SH), 0.55, W_P),
    ("cond_expect", moment(2), soft_gt(1, 60.0, SH), 70.0, W_EF),
    ("cond_expect", moment(2), soft_lt(1, 60.0, SH), 40.0, W_EF),
    ("cov", moment(2), moment(3), -250.0, W_C),
    ("expect", conj(soft_gt(0, 70.0, SH), soft_gt(3, 70.0, SH)), 0.15, W_P),
]
print(f"{len(tn_csts)} constraints in each grammar")

### Fit

Same recipe as the benchmark engines: the chains minimise constraint SSE + a
small entropy regulariser by Adam on exact contractions; the flow minimises
constraint penalties − **exact** joint entropy (`entropy_reg=1.0`) with a fresh
latent batch every step. The bond-1 chain can only represent
`p(x0)p(x1)p(x2)p(x3)` — it is the *independent null* fit to the identical
constraint list.

In [ ]:
fits = {}
for label, bd in (("independent", 1), ("tensor chain", 8)):
    t0 = time.time()
    m = TensorChain(vars_, bond_dim=bd, kind="born")
    loss = combined_loss(m, tn_csts, regularizers=(("entropy", 1e-3),))
    p, h = m.optimize(loss, backend="adam", seed=0,
                      init=m.init_params(seed=0, init="uniform"),
                      steps=2000, lr=2e-2)
    fits[label] = (m, p)
    print(f"{label:<13s} loss {h[0]:.3f} -> {h[-1]:.5f}   ({time.time()-t0:.0f}s)")

t0 = time.time()
fm = FlowSamplerModel(vars_, n_layers=8, hidden=64)
fp, fh = fm.optimize(fm.constraint_loss(fl_csts, n_samples=2048, entropy_reg=1.0),
                     steps=3000, lr=1e-3)
fits["flow"] = (fm, fp)
x_flow = fm.sample(fp, 200_000, seed=7)
print(f"{'flow':<13s} loss {fh[0]:.2f} -> {np.mean(fh[-100:]):.3f}   "
      f"({time.time()-t0:.0f}s)   exact H = {fm.entropy(fp):.2f} nats")

### Constraint satisfaction

Read back with each model's *native* query machinery — exact contractions for
the two chains, 200k continuous samples with **hard** indicators for the flow
(its soft features live only inside the loss). Errors are normalised so mixed
units compare: probabilities as-is, expectations / span, covariance / (span/2)².

In [ ]:
def eval_tc(m, p):
    e2 = m.expectation(p, {2: centers}); e3 = m.expectation(p, {3: centers})
    exy = m.expectation(p, ((2, 3), np.outer(centers, centers)))
    return [
        m.expectation(p, {0: centers}), e2, e3,
        m.event_prob(p, {1: mgt(60)}),
        m.cond_prob(p, {1: mgt(60)}, {0: mgt(70)}),
        m.cond_prob(p, {1: mlt(30)}, {0: mlt(40)}),
        m.cond_expectation(p, 2, {1: mgt(60)}),
        m.cond_expectation(p, 2, {1: mlt(60)}),
        exy - e2 * e3,
        m.event_prob(p, {0: mgt(70), 3: mgt(70)}),
    ]

def eval_flow(x):
    gt = lambda i, t: x[:, i] > t
    return [
        x[:, 0].mean(), x[:, 2].mean(), x[:, 3].mean(),
        gt(1, 60).mean(),
        gt(1, 60)[gt(0, 70)].mean(),
        (~gt(1, 30))[~gt(0, 40)].mean(),
        x[gt(1, 60), 2].mean(),
        x[~gt(1, 60), 2].mean(),
        np.cov(x[:, 2], x[:, 3])[0, 1],
        (gt(0, 70) & gt(3, 70)).mean(),
    ]

LABELS = ["E[x0]=65", "E[x2]=49", "E[x3]=50", "P(x1>60)=0.30",
          "P(x1>60|x0>70)=0.60", "P(x1<30|x0<40)=0.55",
          "E[x2|x1>60]=70", "E[x2|x1<60]=40",
          "cov(x2,x3)=-250", "P(x0>70,x3>70)=0.15"]
TARGETS = [65, 49, 50, 0.30, 0.60, 0.55, 70, 40, -250, 0.15]
NORM = [SPAN, SPAN, SPAN, 1, 1, 1, SPAN, SPAN, 2500, 1]

fitted = {"independent": eval_tc(*fits["independent"]),
          "tensor chain": eval_tc(*fits["tensor chain"]),
          "flow": eval_flow(x_flow)}

print(f"{'constraint':<22s} {'target':>8s} {'indep':>8s} {'tchain':>8s} {'flow':>8s}")
for k, lab in enumerate(LABELS):
    row = [fitted[m][k] for m in COLOR]
    fmt = (lambda v: f"{v:8.3f}") if NORM[k] == 1 else (lambda v: f"{v:8.1f}")
    print(f"{lab:<22s} {fmt(TARGETS[k])} {''.join(fmt(v) for v in row)}")

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.5))
ypos = np.arange(len(LABELS))[::-1]
bh = 0.26
for s, (mname, col) in enumerate(COLOR.items()):
    errs = [abs(fitted[mname][k] - TARGETS[k]) / NORM[k] for k in range(len(LABELS))]
    bars = ax.barh(ypos + (1 - s) * bh, errs, height=bh * 0.92, color=col,
                   label=mname)
    for b, e in zip(bars, errs):
        ax.annotate(f"{e:.3f}", (b.get_width(), b.get_y() + b.get_height() / 2),
                    xytext=(3, 0), textcoords="offset points",
                    va="center", fontsize=7.5, color="#444444")
ax.set_yticks(ypos, LABELS, fontsize=9)
ax.set_xlabel("normalised |error|  (prob as-is, E/span, cov/(span/2)$^2$)")
ax.legend(frameon=False, loc="lower right")
ax.spines[["top", "right"]].set_visible(False)
ax.xaxis.grid(True, color="#dddddd", lw=0.6)
ax.set_axisbelow(True)
plt.tight_layout()
plt.show()

### Joint structure — corner plots

Same layout for all three models: marginals on the diagonal, pairwise
densities below it (each model's own sequential ramp), Pearson correlation
above it. The chains' panels are **exact** 20x20 pair marginals from
contractions; the flow's are 200k samples histogrammed onto the identical grid.

Things to look for: the `x0-x1` and `x1-x2` panels carry the conditional
constraints, `x2-x3` the negative covariance, `x0-x3` the joint tail that the
chain must thread through its bonds — and in the gray figure all of those
panels are exactly rank-one (products of the marginals).

In [ ]:
def pair_grid(mname, i, j):
    if mname == "flow":
        h, _, _ = np.histogram2d(x_flow[:, i], x_flow[:, j], bins=NBINS,
                                 range=[[0, SPAN], [0, SPAN]])
        return h / h.sum()
    m, p = fits[mname]
    return np.asarray(m.joint_marginal(p, (i, j)))

def marg(mname, i):
    if mname == "flow":
        h, _ = np.histogram(x_flow[:, i], bins=NBINS, range=(0, SPAN))
        return h / h.sum()
    m, p = fits[mname]
    return np.asarray(m.joint_marginal(p, (i,)))

def grid_corr(P):
    pi, pj = P.sum(1), P.sum(0)
    ei, ej = pi @ centers, pj @ centers
    c = centers @ P @ centers - ei * ej
    si = np.sqrt(pi @ centers**2 - ei**2); sj = np.sqrt(pj @ centers**2 - ej**2)
    return c / (si * sj)

for mname in COLOR:
    fig, axes = plt.subplots(NV, NV, figsize=(8, 8))
    fig.suptitle(mname, color=COLOR[mname], fontsize=13, fontweight="bold")
    for i in range(NV):
        for j in range(NV):
            ax = axes[i, j]
            if i == j:
                ax.bar(centers, marg(mname, i), width=SPAN / NBINS,
                       color=COLOR[mname])
                ax.set_ylim(0, 0.14)
            elif i > j:               # lower: pair density, axes (x_j, x_i)
                ax.imshow(pair_grid(mname, j, i).T, origin="lower",
                          extent=(0, SPAN, 0, SPAN), cmap=CMAP[mname],
                          aspect="auto")
            else:                     # upper: correlation
                ax.axis("off")
                ax.text(0.5, 0.5, f"r = {grid_corr(pair_grid(mname, i, j)):+.2f}",
                        ha="center", va="center", fontsize=11,
                        transform=ax.transAxes, color="#444444")
                continue
            ax.set_xticks([0, 50, 100] if i == NV - 1 else [])
            ax.set_yticks([0, 50, 100] if j == 0 and i > 0 else [])
            ax.tick_params(labelsize=7)
            if i == NV - 1:
                ax.set_xlabel(names[j], fontsize=9)
            if j == 0 and i > 0:
                ax.set_ylabel(names[i], fontsize=9)
    plt.tight_layout()
    plt.show()

### Notes

- **The independent null is honest about what a product can do.** It matches
  every marginal-expressible target — including, sneakily, the *joint* tail
  #10, which it satisfies by inflating `P(x0>70)` and `P(x3>70)` until their
  product is 0.15 — but its conditionals collapse to marginals
  (`P(x1>60|x0>70) = P(x1>60)`) and its covariance is structurally zero. The
  error chart shows exactly which constraints *require* dependence.
- **The chain fits by exact queries.** Residuals on everything except the
  frustrated-loop tension are optimizer-scale, and its corner panels are exact
  contractions, not histograms. The `x0-x3` dependence really is carried
  through three bonds.
- **The flow's residuals are Monte-Carlo scale** (fresh 2048-sample batch per
  step), and its densities are visibly smoother — the sigmoid squash + entropy
  term rounds off the hard plateau edges the chain can represent. Same story
  as the chain-propagation notebook: agreement on structure, different texture.
- The spec's mild frustration (positive `x0-x1-x2` path, negative `x2-x3`,
  positive `x0-x3`) means no model zeroes every row — compare *where* each one
  chooses to leave its residual.